# Biohub CPU image preparation
Export up to 4 consecutive verified image shards, bounded to 4 GiB for batches, and prepare Gaussian proposals on CPU. No annotation access. Continue the fixed 48/16/48 experiment in resumable stages on the laptop.


In [ ]:
import base64,hashlib,io,os,subprocess,sys,zipfile
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES']=''
os.environ['POLARS_PREFER_PKG']='32'
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='c427b888f5c2526c04ce4a72e43154c6f5230f19bc9038cc4717fc2797df0c11'
package=Path('/kaggle/working/cpu_package');package.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        if not (package/name).resolve().is_relative_to(package.resolve()):raise ValueError(name)
    archive.extractall(package)
support=[Path('/kaggle/input/biohub-tracking-support-pack-50ep-v1'),Path('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1')]
wheels=sorted({str(p.parent) for root in support if root.is_dir() for p in root.rglob('*.whl')})
if not wheels:raise RuntimeError('Offline dependency wheels missing')
cmd=[sys.executable,'-m','pip','install','--no-index']
for folder in wheels:cmd+=['--find-links',folder]
subprocess.run(cmd+['zarr>=3.0.10,<4'],check=True)
env=dict(os.environ,PYTHONPATH=os.pathsep.join([str(package/'src'),str(package/'scripts')]),OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='1')
subprocess.run([sys.executable,'-u',str(package/'scripts/kaggle_cpu_prepare.py'),str(package),'2','4'],env=env,check=True)
